## Libraries

In [114]:
import pandas as pd 
import numpy as np 
import string 
import re 
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from gensim.models import Word2Vec

## Load & Get Data 

In [40]:
data = pd.read_csv('spam.csv',encoding='latin1')

In [41]:
data

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN
...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,NaN,NaN,NaN
5568,ham,Will Ì_ b going to esplanade fr home?,NaN,NaN,NaN
5569,ham,"Pity, * was in mood for that. So...any other s...",NaN,NaN,NaN
5570,ham,The guy did some bitching but I acted like i'd...,NaN,NaN,NaN


## Exploring Data

### General Exploring 

In [42]:
data.shape

(5572, 5)

In [43]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB


In [44]:
data.head(3)

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN


### v1 & v2 Exploring

In [45]:
data['v1'].value_counts()

v1
ham     4825
spam     747
Name: count, dtype: int64

In [46]:
data['v1'].unique()

array(['ham', 'spam'], dtype=object)

In [47]:
data['v1'].isna().sum()

np.int64(0)

In [48]:
# need Balancing 

In [49]:
data['v2'].isna().sum()

np.int64(0)

In [50]:
data['v2'].duplicated().sum()

np.int64(403)

In [51]:
data[data.duplicated(subset=['v1','v2'],keep=False)]

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
7,ham,As per your request 'Melle Melle (Oru Minnamin...,NaN,NaN,NaN
8,spam,WINNER!! As a valued network customer you have...,NaN,NaN,NaN
9,spam,Had your mobile 11 months or more? U R entitle...,NaN,NaN,NaN
11,spam,"SIX chances to win CASH! From 100 to 20,000 po...",NaN,NaN,NaN
...,...,...,...,...,...
5524,spam,You are awarded a SiPix Digital Camera! call 0...,NaN,NaN,NaN
5535,ham,"I know you are thinkin malaria. But relax, chi...",NaN,NaN,NaN
5539,ham,Just sleeping..and surfing,NaN,NaN,NaN
5553,ham,Hahaha..use your brain dear,NaN,NaN,NaN


## Data Cleaning

In [52]:
data_2 = data.drop(columns=['Unnamed: 2' ,'Unnamed: 3' ,'Unnamed: 4'] )

In [54]:
data_2 = data_2.drop_duplicates(subset=['v1','v2'])

In [55]:
data_2

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [57]:
data_2 = data_2.rename(columns={'v2':  'comment'})

In [59]:
data_2 = data_2.rename(columns={'v1' : 'labels'})

In [60]:
data_2

,labels,comment
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [61]:
data_2['labels'].value_counts()

labels
ham     4516
spam     653
Name: count, dtype: int64

## Data Preprocessing 

In [64]:
def preprocess_text(text):
   
    text = text.lower()
    
  
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r'', text)
    
    
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
   
    text = re.sub(r'@\w+', '', text)
    
    
    text = re.sub(r'#\w+', '', text)
    
    
    text = re.sub(r'\d+', '', text)
    
   
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    
    text = re.sub(r'\s+', ' ', text).strip()
    
   
    stop_words = set(stopwords.words('english'))
    stemmer = PorterStemmer()
    
    words = text.split()
    words = [stemmer.stem(word) for word in words if word not in stop_words]
    text = ' '.join(words)
    

    text = ' '.join([word for word in text.split() if len(word) > 2])
    
    return text

data_2['comment_processed'] = data_2['comment'].apply(preprocess_text)


df_final = data_2[['labels', 'comment_processed']].copy()
df_final.rename(columns={'comment_processed': 'comment'}, inplace=True)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


  labels                                            comment
0    ham  jurong point crazi avail bugi great world buff...
1    ham                                   lar joke wif oni
2   spam  free entri wkli comp win cup final tkt may tex...
3    ham                      dun say earli hor alreadi say
4    ham          nah dont think goe usf live around though


In [65]:
df_final

,labels,comment
0,ham,jurong point crazi avail bugi great world buff...
1,ham,lar joke wif oni
2,spam,free entri wkli comp win cup final tkt may tex...
3,ham,dun say earli hor alreadi say
4,ham,nah dont think goe usf live around though
...,...,...
5567,spam,time tri contact pound prize claim easi call p...
5568,ham,esplanad home
5569,ham,piti mood soani suggest
5570,ham,guy bitch act like interest buy someth els nex...


In [66]:
df_final.duplicated().sum()

np.int64(147)

In [68]:
df_cleaned = df_final.groupby('comment', as_index=False).agg({
    'labels': lambda x: 'spam' if (x == 'spam').sum() >= (x == 'ham').sum() else 'ham'
})

In [69]:
df_cleaned

,comment,labels
0,,ham
1,aah bless how arm,ham
2,aah cuddl would lush need lot tea soup kind fumbl,ham
3,aaooooright work,ham
4,aathiwher dear,ham
...,...,...
5017,yup wun believ wat realli neva msg sent shuhui,ham
5018,yupz ive oredi book slot weekend liao,ham
5019,ywhere dogbreath sound like jan thatåõ,ham
5020,zoe hit fuck shitin defo tri hardest cum morow...,ham


In [71]:
df_cleaned.duplicated().sum()

np.int64(0)

### Shuffle & Reset index

In [72]:
df_cleaned = df_cleaned.sample(frac=1, random_state=42).reset_index(drop=True)

In [74]:
df_cleaned

,comment,labels
0,good morn dear great amp success day,ham
1,catch fri egg make tea eat mom left dinner fee...,ham
2,sent,ham
3,yesterday true true,ham
4,would like see xxx pic hot nearli ban,spam
...,...,...
5017,urgent urgent free flight europ give away call...,spam
5018,call senthil hsbc,ham
5019,packag program well,ham
5020,sorri took long omw,ham


## Feature Extraction

### TD-IDF N-gram

In [79]:
X = df_cleaned['comment']
y = df_cleaned['labels']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 3),
    stop_words=None,          
    max_df=0.7,
    min_df=3,
    sublinear_tf=True,
    use_idf=True,
    smooth_idf=True
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


## Train model

### Logestic regression

In [96]:
model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    C=1.0,                    
    solver='liblinear',       
    random_state=42
)

model.fit(X_train_vec, y_train)
y_pred = model.predict(X_test_vec)

print(classification_report(y_test, y_pred))
print(f"F1-Score: {f1_score(y_test, y_pred, pos_label='spam'):.4f}")

print(f"\nماتریس درهم‌ریختگی:\n{confusion_matrix(y_test, y_pred)}")

              precision    recall  f1-score   support

         ham       0.98      0.99      0.99       889
        spam       0.91      0.87      0.89       116

    accuracy                           0.98      1005
   macro avg       0.95      0.93      0.94      1005
weighted avg       0.97      0.98      0.97      1005

F1-Score: 0.8899

ماتریس درهم‌ریختگی:
[[879  10]
 [ 15 101]]


### Random forest

In [83]:
model = RandomForestClassifier(
    n_estimators=100,         
    max_depth=20,             
    class_weight='balanced',
    random_state=42,
    n_jobs=-1                  
)

model.fit(X_train_vec, y_train)
y_pred = model.predict(X_test_vec)

print(classification_report(y_test, y_pred))
print(f"F1-Score: {f1_score(y_test, y_pred, pos_label='spam'):.4f}")

              precision    recall  f1-score   support

         ham       0.97      1.00      0.99       889
        spam       1.00      0.78      0.87       116

    accuracy                           0.97      1005
   macro avg       0.99      0.89      0.93      1005
weighted avg       0.97      0.97      0.97      1005

F1-Score: 0.8738


### XGBoost

In [102]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)


scale_pos_weight = (y_train_enc == 0).sum() / (y_train_enc == 1).sum()

model = XGBClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

model.fit(X_train_vec, y_train_enc)
y_pred = model.predict(X_test_vec)


y_pred_labels = le.inverse_transform(y_pred)

print(classification_report(y_test, y_pred_labels))
print(f"F1-Score: {f1_score(y_test_enc, y_pred, pos_label=1):.4f}")

C:\Users\LENOVO\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [12:12:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


              precision    recall  f1-score   support

         ham       0.97      0.98      0.98       889
        spam       0.82      0.80      0.81       116

    accuracy                           0.96      1005
   macro avg       0.90      0.89      0.89      1005
weighted avg       0.96      0.96      0.96      1005

F1-Score: 0.8122


### best hyperparametr fir logestic regression 

In [97]:
param_grid = {
    'C': [0.1, 0.5, 1.0, 2.0],
    'solver': ['liblinear', 'saga'],
    'max_iter': [1000, 2000]
}

grid = GridSearchCV(
    LogisticRegression(class_weight='balanced', random_state=42),
    param_grid,
    cv=5,
    scoring='f1_macro'
)

grid.fit(X_train_vec, y_train)
print(f"بهترین پارامترها: {grid.best_params_}")
print(f"بهترین F1: {grid.best_score_:.4f}")

# مدل نهایی با بهترین پارامترها
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test_vec)

بهترین پارامترها: {'C': 2.0, 'max_iter': 1000, 'solver': 'liblinear'}
بهترین F1: 0.9337


In [107]:
best_model = LogisticRegression(
    C=2.0,
    max_iter=1000,
    solver='liblinear',
    class_weight='balanced',
    random_state=42
)

best_model.fit(X_train_vec, y_train)
y_pred_tuned = best_model.predict(X_test_vec)


print("=== مدل بهینه شده ===")
print(classification_report(y_test, y_pred_tuned))
print(f"F1-Score (Spam): {f1_score(y_test, y_pred_tuned, pos_label='spam'):.4f}")
print(f"\nماتریس درهم‌ریختگی:\n{confusion_matrix(y_test, y_pred_tuned)}")

=== مدل بهینه شده ===
              precision    recall  f1-score   support

         ham       0.98      0.99      0.99       889
        spam       0.91      0.87      0.89       116

    accuracy                           0.98      1005
   macro avg       0.95      0.93      0.94      1005
weighted avg       0.97      0.98      0.97      1005

F1-Score (Spam): 0.8899

ماتریس درهم‌ریختگی:
[[879  10]
 [ 15 101]]


## save model 

In [111]:
desktop_path = os.path.join(os.path.expanduser('~'),'OneDrive' , 'Desktop')
model_path = os.path.join(desktop_path, 'spam_model.pkl')
vec_path = os.path.join(desktop_path, 'vectorizer.pkl')


joblib.dump(best_model, model_path)
joblib.dump(vectorizer, vec_path)

print(f"✅ مدل در: {model_path}")
print(f"✅ وکتورایزر در: {vec_path}")


loaded_model = joblib.load(model_path)
loaded_vec = joblib.load(vec_path)
print("✅ بارگذاری مجدد موفق بود!")


sample = ["win free money now"]
sample_vec = loaded_vec.transform(sample)
pred = loaded_model.predict(sample_vec)
print(f"پیش‌بینی: {pred[0]}")

✅ مدل در: C:\Users\LENOVO\OneDrive\Desktop\spam_model.pkl
✅ وکتورایزر در: C:\Users\LENOVO\OneDrive\Desktop\vectorizer.pkl
✅ بارگذاری مجدد موفق بود!
پیش‌بینی: spam


## another feature extract and diff with tf_idf 

In [116]:
X = df_cleaned['comment'].values
y = df_cleaned['labels'].values


sentences = [comment.split() for comment in X]


w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,        
    window=5,
    min_count=2,
    workers=4,
    sg=1,                   
    seed=42
)

print(f"واژگان یاد گرفته شده: {len(w2v_model.wv)} کلمه")


def get_comment_vector(tokens, model, vector_size=100):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)


X_w2v = np.array([get_comment_vector(comment.split(), w2v_model) for comment in X])

print(f"شکل داده‌های Word2Vec: {X_w2v.shape}")


X_train, X_test, y_train, y_test = train_test_split(
    X_w2v, y, test_size=0.2, random_state=42, stratify=y
)


le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)


model_w2v = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

model_w2v.fit(X_train, y_train_enc)
y_pred = model_w2v.predict(X_test)


print("\n=== Word2Vec + Logistic Regression ===")
print(classification_report(y_test_enc, y_pred, target_names=le.classes_))
print(f"F1-Score (Spam): {f1_score(y_test_enc, y_pred, pos_label=1):.4f}")

واژگان یاد گرفته شده: 2796 کلمه
شکل داده‌های Word2Vec: (5022, 100)

=== Word2Vec + Logistic Regression ===
              precision    recall  f1-score   support

         ham       0.98      0.94      0.96       889
        spam       0.64      0.84      0.73       116

    accuracy                           0.93      1005
   macro avg       0.81      0.89      0.85      1005
weighted avg       0.94      0.93      0.93      1005

F1-Score (Spam): 0.7313
